# 现代数据探索

用 Polars 做基础探索。

In [ ]:
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import polars.selectors as cs

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "data").is_dir() and (candidate / "pyproject.toml").is_file():
        DATA_DIR = candidate / "data"
        break
else:
    raise FileNotFoundError("请从 mining 或其子目录启动内核")

RANDOM_STATE = 42

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


## 缺失值

统计空值、全空列和全空行。

In [ ]:
sample = pl.DataFrame(
    {
        "A": [1.0, None, 3.0, None],
        "B": [None, None, None, None],
        "C": ["x", "y", None, None],
        "D": [None, None, None, None],
    }
)

missing_summary = pl.DataFrame(
    [
        {
            "column": column,
            "missing_count": sample[column].null_count(),
            "missing_rate": sample[column].null_count() / sample.height,
        }
        for column in sample.columns
    ]
)
all_missing_columns = [column for column in sample.columns if sample[column].null_count() == sample.height]
all_missing_rows = (
    sample.with_row_index("row_id")
    .filter(pl.all_horizontal(pl.exclude("row_id").is_null()))
    .get_column("row_id")
    .to_list()
)

display(sample)
display(missing_summary)
print(f"全为空值的列: {all_missing_columns}")
print(f"全为空值的行号: {all_missing_rows}")


## 异常值

用 IQR 规则识别异常点。

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
values = pl.Series("value", rng.normal(loc=0, scale=1, size=1000))
q1 = values.quantile(0.25)
q3 = values.quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
outliers = values.filter((values < lower_bound) | (values > upper_bound))

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.boxplot(values.to_numpy(), vert=True, patch_artist=True, showmeans=True)
ax.scatter(np.ones(len(outliers)), outliers.to_numpy(), color="crimson", s=18, label="异常点")
ax.set_title("正态分布样本箱线图")
ax.set_ylabel("取值")
ax.legend()
plt.show()

print(f"异常值数量: {len(outliers)}")
print(f"异常值边界: [{lower_bound:.3f}, {upper_bound:.3f}]")


## 描述统计

用 Lazy API 读取并汇总成绩数据。

In [ ]:
scores_lazy = pl.scan_csv(DATA_DIR / "scores.csv")
scores = scores_lazy.collect()
score_summary = scores.select(cs.numeric()).describe()
score_extra = scores_lazy.select(
    [
        (pl.col(column).max() - pl.col(column).min()).alias(f"{column}_range")
        for column in scores.select(cs.numeric()).columns
    ]
    + [
        (pl.col(column).std() / pl.col(column).mean()).alias(f"{column}_cv")
        for column in scores.select(cs.numeric()).columns
    ]
).collect()

display(scores.head())
display(score_summary)
display(score_extra)


## 相关性

计算广告投入和销售额相关性。

In [ ]:
turnover = pl.read_csv(DATA_DIR / "turnover.csv")
correlation = turnover.select(pl.corr("年广告费投入", "月均销售额").alias("corr"))

fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
ax.scatter(turnover["年广告费投入"].to_numpy(), turnover["月均销售额"].to_numpy())
ax.set_title("广告费投入与月均销售额")
ax.set_xlabel("年广告费投入")
ax.set_ylabel("月均销售额")
ax.grid(alpha=0.25)
plt.show()

display(turnover)
display(correlation)
